# 🔬 Customer Health Forensics System
## Phase 1 — Data Pipeline + ML Model Selection

**Runtime:** Google Colab T4 GPU  
**Expected time:** ~8–12 min (full 500k run)  
**What this notebook does:**
1. Installs dependencies
2. Mounts Google Drive (optional — for persisting artifacts)
3. Uploads project source files
4. Generates 500k synthetic customer records
5. Engineers modular features
6. Trains + selects best model (XAI-coupled logic)
7. Saves all artifacts to Drive

---
> **After Phase 1 completes**, open `Phase2_XAI.ipynb` to build the Consensus XAI Engine.

## ⚙️ Step 1 — Install Dependencies

In [ ]:
# All packages needed for Phase 1
# XGBoost + imbalanced-learn are the only installs beyond Colab defaults
!pip install xgboost imbalanced-learn --quiet

import subprocess, sys
pkgs = ['numpy', 'pandas', 'sklearn', 'xgboost', 'matplotlib', 'joblib']
for p in pkgs:
    try:
        __import__(p if p != 'sklearn' else 'sklearn')
        print(f'  ✓ {p}')
    except ImportError:
        print(f'  ✗ {p} — MISSING')
print('\nDependencies ready.')

## 📁 Step 2 — Mount Google Drive (Recommended)

In [ ]:
# Mount Drive to persist all artifacts beyond this Colab session.
# If you skip this, outputs will be lost when the session ends.
# Set USE_DRIVE = False to skip.

USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/CustomerHealthForensics'
    print(f'Using Drive: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '/content/CustomerHealthForensics'
    print(f'Using local Colab storage: {PROJECT_ROOT}')

import os
os.makedirs(f'{PROJECT_ROOT}/data',    exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/models',  exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)
os.makedirs(f'{PROJECT_ROOT}/src',     exist_ok=True)
print('Directories created.')

## 📤 Step 3 — Upload Source Files

In [ ]:
# Upload all 4 source files from the ZIP you downloaded:
#   data_generator.py
#   feature_engineering.py
#   model_selector.py
#   pipeline.py

from google.colab import files
print('Select all 4 .py files from the src/ folder of the ZIP:')
uploaded = files.upload()

import shutil
for fname in uploaded:
    dest = f'{PROJECT_ROOT}/src/{fname}'
    with open(dest, 'wb') as f:
        f.write(uploaded[fname])
    print(f'  Saved: {dest}')

import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')
print('\nSource files ready.')

## 🏃 Step 4 — Quick Smoke Test (10k rows)
Run this FIRST to verify everything works before the full 500k run.

In [ ]:
import sys
sys.path.insert(0, f'{PROJECT_ROOT}/src')

from pipeline import run_phase1
from pathlib import Path

print('Running smoke test (10k rows, ~60 seconds)...')
summary = run_phase1(
    data_dir    = Path(f'{PROJECT_ROOT}/data'),
    models_dir  = Path(f'{PROJECT_ROOT}/models'),
    output_dir  = Path(f'{PROJECT_ROOT}/outputs'),
    sample_size = 10_000,        # fast smoke test
)

print('\n✓ Smoke test passed. Results:')
print(f"  Best model : {summary['best_model_name']}")
print(f"  Val AUC    : {summary['val_auc']:.4f}")
print(f"  Val F1     : {summary['val_f1']:.4f}")
print(f"  XAI method : {summary['xai_method']}")

## 🚀 Step 5 — Full 500k Run
**Expected time on T4: ~8–12 minutes**  
Only run this after the smoke test passes.

In [ ]:
import importlib, pipeline
importlib.reload(pipeline)  # reload in case smoke test ran in same session
from pipeline import run_phase1
from pathlib import Path

print('Starting FULL 500k run...')
print('(XGBoost + Logistic trained, size-aware selection, XAI-coupled tiebreaker)')

summary = run_phase1(
    data_dir    = Path(f'{PROJECT_ROOT}/data'),
    models_dir  = Path(f'{PROJECT_ROOT}/models'),
    output_dir  = Path(f'{PROJECT_ROOT}/outputs'),
    # No sample_size = full 500k run
)

import json
print('\n' + '='*50)
print('PHASE 1 COMPLETE')
print('='*50)
print(json.dumps(summary, indent=2))

## 📊 Step 6 — View Results

In [ ]:
import pandas as pd
from pathlib import Path

models_dir  = Path(f'{PROJECT_ROOT}/models')
outputs_dir = Path(f'{PROJECT_ROOT}/outputs')

# Leaderboard
print('── Model Leaderboard ──────────────────────────────')
lb = pd.read_csv(models_dir / 'leaderboard.csv')
print(lb[['name','val_auc','val_f1','test_auc','test_f1','selected','train_time_s']].to_string(index=False))

# Logistic coefficients
print('\n── Logistic Regression Coefficients (top 10) ─────')
coefs = pd.read_csv(models_dir / 'logistic_coefficients.csv')
print(coefs[['feature','coefficient','direction']].head(10).to_string(index=False))

# Evaluation plot
from IPython.display import Image
Image(str(outputs_dir / 'phase1_evaluation.png'))

## 🔍 Step 7 — Inspect Feature Registry

In [ ]:
from feature_engineering import get_feature_names, get_feature_docs

docs = get_feature_docs()
print(f'Registered engineered features: {len(docs)}')
print()
for name, desc in docs.items():
    print(f'  {name}')
    print(f'    → {desc}')
    print()

## ➕ Optional — Add a Custom Feature
Demonstrates extensibility. No downstream code changes required.

In [ ]:
from feature_engineering import register_feature, get_feature_names
import numpy as np
import pandas as pd

# Example: add a referral_efficiency feature
@register_feature(
    'referral_efficiency',
    'referrals_made / tenure_months — how actively a customer promotes the product'
)
def _referral_efficiency(df: pd.DataFrame) -> pd.Series:
    return (df['referrals_made'] / df['tenure_months'].clip(lower=1)).round(4)

print(f'Features now registered: {len(get_feature_names())}')
print(f'New feature included: {"referral_efficiency" in get_feature_names()}')
print()
print('To use it in a re-run: the pipeline picks it up automatically.')
print('No other files need to change.')

---
## ✅ Phase 1 Complete

**Artifacts saved to Drive:**
```
models/
  best_model.pkl              ← fitted model (Phase 2 XAI will load this)
  best_model_info.json        ← selection metadata + XAI method
  leaderboard.csv             ← all models + scores
  logistic_coefficients.csv   ← LR weights (XAI reference)
  feature_names.json          ← feature list for Phase 2
  phase1_summary.json         ← full pipeline summary

data/
  customers.csv               ← 500k records (~43 MB)
  customers_snapshots.csv     ← 6M rows, 12-month snapshots (drift)

outputs/
  phase1_evaluation.png       ← ROC + confusion matrix + leaderboard
  classification_report.txt   ← precision/recall/F1 per class
```

**Next:** Open `Phase2_XAI.ipynb` — Consensus XAI Engine (SHAP + LIME + AIX360)